# S6_03 — MCP 클라이언트: `MCPClient` 클래스와 async 컨텍스트 매니저

**Skilljar Lessons**: L02 (MCP 클라이언트) · L06 (클라이언트 구현)

## 강의노트 매핑 (`Week_07.md`)

| 절 | 주제 | 라인 |
|---|---|---|
| §1.2 | 클라이언트 아키텍처, 전송 방식 무관성, 시퀀스 다이어그램 | 312-435 |
| §2.1 | `MCPClient` 클래스, 메서드 `list_tools` 와 `call_tool`, 구문 `async with` | 767-894 |

## 사전 준비
본 노트북을 실행하기 전에 다음을 확인한다.
- 본 폴더에 `mcp_server.py` 가 저장되어 있어야 한다 (S6_01 의 §10 에서 저장).
- 명령 `pip install "mcp[cli]" anthropic python-dotenv pydantic` 으로 패키지가 모두 설치되어 있어야 한다.
- 선택 사항으로, `uv` 가 설치되어 있다면 인자 `command="uv", args=["run", "mcp_server.py"]` 를 사용한다. 없다면 `command="python", args=["mcp_server.py"]` 로 대체한다.
- §6 의 마지막 통합 예제를 실행하려면 환경 변수 `ANTHROPIC_API_KEY` 가 `.env` 파일에 설정되어 있어야 한다.

## 학습 목표
본 노트북을 마치면 다음을 할 수 있어야 한다.
1. **두 컴포넌트** 로 이루어진 클라이언트 구조 — 우리의 `MCPClient` 래퍼 클래스와 SDK 가 제공하는 `ClientSession` — 를 이해한다.
2. 메서드 `__init__`, `connect`, `session`, `list_tools`, `call_tool` 을 차례로 구현한다.
3. 메서드 `__aenter__` 와 `__aexit__` 로 감싸 구문 `async with` 가 stdio 서브프로세스를 자동으로 정리하게 만든다.
4. 위에서 만든 클라이언트를 Claude 의 도구 사용 루프에 끼워 넣는다 (S6_04 와 S6_05 패턴 미리 보기).


## §1. 클라이언트 아키텍처 — 전송 방식과 무관한 통신

MCP 의 큰 강점 중 하나는 클라이언트와 서버 사이의 **통신 방식이 고정되어 있지 않다는 점** 이다. 같은 머신 안에서 동작하는 가장 흔한 셋업은 stdio 다 — 클라이언트가 서버를 자식 프로세스로 띄우고, 표준 입력과 표준 출력 위로 JSON-RPC 메시지를 주고받는다. 그러나 같은 클라이언트 코드가 HTTP 나 WebSocket 같은 네트워크 프로토콜로도 서버에 연결될 수 있다.

메시지의 종류는 명세에 정해져 있지만, 실무에서 가장 자주 다루게 되는 것은 다음 두 짝이다.

| 요청 메시지 | 결과 메시지 | 용도 |
|---|---|---|
| `ListToolsRequest` | `ListToolsResult` | 서버가 어떤 도구를 제공하는지 발견 |
| `CallToolRequest` | `CallToolResult` | 특정 도구를 인자와 함께 실행 |

강의노트 §1.2 의 라인 398 부근에 그려진 시퀀스 다이어그램이 전체 흐름을 시각화한다. 사용자의 질문에서 시작해 우리 앱, MCP 클라이언트, MCP 서버, 외부 API 까지 갔다가 같은 경로를 역순으로 되돌아오는 여정이다. 우리가 본 노트북에서 만들 `MCPClient` 는 그 여정의 **중간 다리** 역할을 한다 — 메시지 직렬화와 서브프로세스 관리를 모두 클래스 안에 숨기고, 사용자에게는 두 개의 깨끗한 메서드만 노출한다.

> [!tip] 강의노트 §1.2 인용 — 전송 방식 무관성
> 클라이언트와 서버가 다양한 통신 방법으로 서로 대화할 수 있다는 뜻이다. 가장 흔한 설정은 같은 머신에서 실행되며 표준 입출력으로 통신하는 것이다.


## §2. 셋업 — Skilljar 원본과 동일한 import 묶음

본 노트북의 imports 는 Skilljar 원본 파일의 처음 여덟 줄을 그대로 복사한 것이다. 외부 패키지로는 Pydantic 의 URI 검증용 `AnyUrl`, 표준 라이브러리에서는 비동기 자원 관리를 위한 `AsyncExitStack`, 그리고 MCP SDK 본체에서는 세션 클래스와 stdio 클라이언트 함수가 등장한다. 이 묶음 자체가 "MCP 클라이언트를 만들 때 필요한 최소 도구" 라고 보아도 무방하다.


In [ ]:
# cli_project/mcp_client.py:1-8 — Skilljar 원본과 동일한 import 들
import sys
import asyncio
import json
from pydantic import AnyUrl
from typing import Optional, Any
from contextlib import AsyncExitStack
from mcp import ClientSession, StdioServerParameters, types
from mcp.client.stdio import stdio_client


## §3. 클래스의 골격 — 메서드 `__init__`, `connect`, `session`

본 절에서 만들 클래스 `MCPClient` 의 책임은 셋이다.

첫째, 메서드 `__init__` 은 서버를 어떻게 띄울지 — 명령어, 인자, 환경 변수 — 를 단순히 저장만 한다. 실제로 서브프로세스를 띄우는 일은 이 단계에서 하지 않는다. 그 이유는 다음 단계의 비동기 컨텍스트로 넘긴다.

둘째, 메서드 `connect` 는 SDK 의 `stdio_client` 로 서브프로세스를 띄우고, 그 입출력 스트림 위에 `ClientSession` 을 연 뒤, 메서드 `initialize()` 를 호출해 프로토콜 핸드셰이크까지 완료한다. 이 모든 자원의 정리가 자동으로 이루어지도록 `AsyncExitStack` 을 사용해 트랜잭션처럼 묶어 둔다.

셋째, 메서드 `session()` 은 매우 작은 접근자다. 사용자가 메서드 `connect()` 호출을 깜빡한 채 도구를 부르려 하면, 명확한 오류 메시지와 함께 예외를 던진다. 디버깅 시간을 크게 줄여 주는 작은 안전장치다.


In [ ]:
# cli_project/mcp_client.py:11-44 — 클래스 골격 (원본 그대로)
# Week_07.md §2.1 라인 ~767-800
class MCPClient:
    def __init__(
        self,
        command: str,
        args: list[str],
        env: Optional[dict] = None,
    ):
        self._command = command
        self._args = args
        self._env = env
        self._session: Optional[ClientSession] = None
        self._exit_stack: AsyncExitStack = AsyncExitStack()

    async def connect(self):
        server_params = StdioServerParameters(
            command=self._command,
            args=self._args,
            env=self._env,
        )
        stdio_transport = await self._exit_stack.enter_async_context(
            stdio_client(server_params)
        )
        _stdio, _write = stdio_transport
        self._session = await self._exit_stack.enter_async_context(
            ClientSession(_stdio, _write)
        )
        await self._session.initialize()

    def session(self) -> ClientSession:
        if self._session is None:
            raise ConnectionError(
                "Client session not initialized or cache not populated. Call connect_to_server first."
            )
        return self._session


print("MCPClient skeleton defined.")


## §4. 핵심 메서드 — `list_tools` 와 `call_tool`

본 절의 두 메서드는 §1 에서 본 두 메시지 짝과 정확히 일대일로 대응한다. 메서드 `list_tools` 는 `ListToolsRequest` 를 보내 결과를 받고, 메서드 `call_tool` 은 `CallToolRequest` 를 보낸다. 둘 다 SDK 의 `ClientSession` 인스턴스에 그저 위임할 뿐이며, 우리 클래스의 임무는 그 결과의 형태를 도메인 코드가 쓰기 좋게 다듬는 것이다.

이렇게 메서드 이름이 메시지 짝의 의미를 그대로 따라가도록 만들어 두면, 클라이언트 코드를 읽는 사람은 "이 호출이 선로 위로 어떤 메시지를 흘려보내는지" 를 한눈에 짐작할 수 있다.

> [!finding] 강의노트 §2.1 인용 — 다리로서의 클라이언트
> 연결의 세부 사항은 모두 클라이언트가 감추므로, 우리 도메인 코드는 `await client.list_tools()` 와 `await client.call_tool(...)` 같은 도메인 로직 수준의 호출만 신경 쓰면 된다.


In [ ]:
# cli_project/mcp_client.py:46-55 — list_tools + call_tool (원본 그대로)
# Week_07.md §2.1 라인 ~810-826
async def _list_tools(self) -> list[types.Tool]:
    result = await self.session().list_tools()
    return result.tools


async def _call_tool(
    self, tool_name: str, tool_input: dict
) -> types.CallToolResult | None:
    return await self.session().call_tool(tool_name, tool_input)


# MCPClient 에 부착 (서술 흐름을 위해 셀을 분리했지만, 최종 mcp_client.py 파일에서는
# 클래스의 일반 메서드로 들어 있다).
MCPClient.list_tools = _list_tools
MCPClient.call_tool = _call_tool

print("list_tools and call_tool attached.")


## §5. 비동기 컨텍스트 매니저 — `cleanup`, `__aenter__`, `__aexit__`

Python 표준 라이브러리의 `AsyncExitStack` 은 여러 비동기 자원의 진입과 정리를 한 묶음으로 다룰 수 있게 해 준다. 본 클래스에서는 stdio 전송 계층과 세션 객체 두 가지를 그 스택에 등록해 두고, 메서드 `cleanup` 안의 호출 한 번 — `aclose()` — 으로 등록 역순으로 정리한다.

이 구조를 메서드 `__aenter__` 와 `__aexit__` 로 감싸면, 클라이언트의 사용자는 단 한 줄의 `async with` 블록만으로 **연결, 사용, 정리** 의 전체 수명 주기를 안전하게 관리할 수 있다. 블록 안에서 예외가 나더라도 메서드 `__aexit__` 가 호출되어 서브프로세스가 좀비로 남지 않는다. 이는 작은 차이처럼 보이지만 장기 실행 서비스에서는 매우 중요한 안전장치다.

> [!tip] 강의노트 §2.1 인용 — 구문 `async with` 의 의미
> `MCPClient` 는 비동기 컨텍스트 매니저로 설계되어 있다. 메서드 `__aenter__` 에서 서버 서브프로세스 기동과 stdio 연결을, 메서드 `__aexit__` 에서 안전한 종료와 자원 해제를 수행한다.


In [ ]:
# cli_project/mcp_client.py:77-86 — cleanup, __aenter__, __aexit__ (원본 그대로)
# Week_07.md §2.1 라인 ~830-846
async def _cleanup(self):
    await self._exit_stack.aclose()
    self._session = None


async def _aenter(self):
    await self.connect()
    return self


async def _aexit(self, exc_type, exc_val, exc_tb):
    await self.cleanup()


MCPClient.cleanup = _cleanup
MCPClient.__aenter__ = _aenter
MCPClient.__aexit__ = _aexit

print("MCPClient is now a full async context manager.")


## §6. `MCPClient` 사용 — 구문 `async with` 사용 예

이전 노트북 S6_01 이 저장한 `mcp_server.py` 를 stdio 서브프로세스로 띄운 뒤, 도구 목록을 가져오고 도구 `read_doc_contents` 를 호출한다. 블록 `async with` 가 끝나면 서브프로세스가 자동으로 정리된다. 이전 노트북 S6_02 에서 손으로 짠 JSON-RPC 메시지 흐름이, 본 절에서는 단 하나의 메서드 호출 뒤에 깔끔하게 숨는다.

환경에 맞는 실행 명령을 골라야 한다. 도구 `uv` 가 설치되어 있다면 인자 묶음 `command="uv", args=["run", "mcp_server.py"]` 를 쓴다. 그렇지 않다면 인자 묶음 `command="python", args=["mcp_server.py"]` 가 동등하게 동작한다. 아래 셀은 명령 `which uv` 로 자동 감지를 시도하므로, 둘 중 어느 환경에서도 같은 코드가 그대로 동작한다.


In [ ]:
# Week_07.md §2.1 라인 ~836 — async with 로 왕복
import os

# uv 가 있는지 자동 감지
USE_UV = os.system("which uv > /dev/null 2>&1") == 0
if USE_UV:
    cmd, args = "uv", ["run", "mcp_server.py"]
else:
    cmd, args = "python", ["mcp_server.py"]

print(f"실행 명령: {cmd} {' '.join(args)}")

async with MCPClient(command=cmd, args=args) as client:
    tools = await client.list_tools()
    print("\n서버가 광고하는 도구 목록:")
    for t in tools:
        print(f"  - {t.name}: {t.description}")

    print("\nread_doc_contents('plan.md') 호출 중...")
    result = await client.call_tool(
        "read_doc_contents", {"doc_id": "plan.md"}
    )
    print("결과:", result)


## §7. Claude 와의 통합 — 도구 사용 루프 미리보기

본 절은 본 노트북에서 가장 실전에 가까운 코드다. 서버가 광고하는 모든 도구가 Claude 의 인자 `tools` 에 그대로 전달되어, **4주차에서 배운 도구 사용 루프** 를 그대로 사용할 수 있다. 단 하나의 차이는 "도구 함수의 본체를 우리가 직접 작성하지 않는다" 는 점뿐이다. 도구의 정의와 실행은 모두 MCP 서버가 책임지고, 우리 앱은 단지 **모델의 선택 결과를 받아 클라이언트로 전달** 하기만 한다.

아래 셀의 함수 `chat_with_mcp` 는 그 흐름의 최소 형태를 보여준다. 사용자의 메시지를 모델에게 보내고, 응답이 도구 호출을 요청하면 우리 클라이언트로 도구를 실행한다. 그 결과를 다시 모델에게 돌려보내 최종 답변까지 받아내는 표준 루프다. 다음 노트북 S6_04 가 그 위에 리소스 처리를 더하고, 그다음 노트북 S6_05 가 프롬프트 지원을 더하면, 사용자가 슬래시 명령 `/format plan.md` 같은 호출로 서버의 능력을 직접 활용하는 형태까지 자연스럽게 발전한다.


In [ ]:
# Week_07.md §2.1 라인 ~860 — 전체 앱 흐름 미리보기
import anthropic
from dotenv import load_dotenv

load_dotenv()
MODEL = "claude-haiku-4-5"


async def chat_with_mcp(user_message: str, max_iters: int = 5):
    """MCP 서버가 구동하는 tool-use 루프. 기본 패턴은 Week_04 참조."""
    api = anthropic.Anthropic()

    async with MCPClient(command=cmd, args=args) as client:
        # MCP 서버에서 도구 스키마를 가져온다 (직접 손으로 작성하지 않는다).
        mcp_tools = await client.list_tools()
        claude_tools = [
            {"name": t.name, "description": t.description, "input_schema": t.inputSchema}
            for t in mcp_tools
        ]

        messages = [{"role": "user", "content": user_message}]

        for _ in range(max_iters):
            resp = api.messages.create(
                model=MODEL,
                max_tokens=1024,
                tools=claude_tools,
                messages=messages,
            )
            messages.append({"role": "assistant", "content": resp.content})

            if resp.stop_reason == "end_turn":
                return "".join(b.text for b in resp.content if hasattr(b, "text"))

            if resp.stop_reason == "tool_use":
                tool_results = []
                for block in resp.content:
                    if block.type == "tool_use":
                        tool_out = await client.call_tool(block.name, block.input)
                        # CallToolResult 에서 텍스트 추출
                        text = (
                            tool_out.content[0].text
                            if tool_out and tool_out.content
                            else str(tool_out)
                        )
                        tool_results.append({
                            "type": "tool_result",
                            "tool_use_id": block.id,
                            "content": text,
                        })
                messages.append({"role": "user", "content": tool_results})
                continue

            break

        return "<no end_turn within max_iters>"


# 끝까지 실행하려면 주석 해제 (ANTHROPIC_API_KEY 필요):
# answer = await chat_with_mcp("plan.md 의 내용은? 그리고 'outlines' 를 'describes' 로 바꿔줘.")
# print(answer)
print("chat_with_mcp 정의 완료. 위 호출의 주석을 풀면 루프를 직접 체험할 수 있다.")


## §8. 다음 노트북을 위해 `mcp_client.py` 저장하기

다음 노트북 S6_04 는 메서드 `read_resource` 를 추가하고, 그다음 노트북 S6_05 는 메서드 `list_prompts` 와 `get_prompt` 를 추가한다. 그 노트북들이 단순한 import 한 줄 — `from mcp_client import MCPClient` — 으로 본 클래스를 그대로 가져다 쓸 수 있도록, **완성된 형태의** Skilljar `mcp_client.py` 를 본 셀에서 그대로 디스크에 저장한다.

본 노트북에서는 서술의 흐름을 위해 메서드들을 여러 셀에 나누어 클래스에 차례로 부착했지만, 디스크에 저장되는 파일에서는 모든 메서드가 하나의 클래스 정의 안에 자연스럽게 모여 있는 표준 형태다. 이렇게 하면 다음 노트북들이 본 노트북의 셀 분할 흔적과 무관하게, 깨끗한 단일 클래스로 모든 기능을 한 번에 가져다 쓸 수 있다.


In [ ]:
import os as _os

CLIENT_CODE = '''import sys
import asyncio
import json
from pydantic import AnyUrl
from typing import Optional, Any
from contextlib import AsyncExitStack
from mcp import ClientSession, StdioServerParameters, types
from mcp.client.stdio import stdio_client


class MCPClient:
    def __init__(
        self,
        command: str,
        args: list[str],
        env: Optional[dict] = None,
    ):
        self._command = command
        self._args = args
        self._env = env
        self._session: Optional[ClientSession] = None
        self._exit_stack: AsyncExitStack = AsyncExitStack()

    async def connect(self):
        server_params = StdioServerParameters(
            command=self._command,
            args=self._args,
            env=self._env,
        )
        stdio_transport = await self._exit_stack.enter_async_context(
            stdio_client(server_params)
        )
        _stdio, _write = stdio_transport
        self._session = await self._exit_stack.enter_async_context(
            ClientSession(_stdio, _write)
        )
        await self._session.initialize()

    def session(self) -> ClientSession:
        if self._session is None:
            raise ConnectionError(
                "Client session not initialized or cache not populated. Call connect_to_server first."
            )
        return self._session

    async def list_tools(self) -> list[types.Tool]:
        result = await self.session().list_tools()
        return result.tools

    async def call_tool(
        self, tool_name: str, tool_input: dict
    ) -> types.CallToolResult | None:
        return await self.session().call_tool(tool_name, tool_input)

    async def list_prompts(self) -> list[types.Prompt]:
        result = await self.session().list_prompts()
        return result.prompts

    async def get_prompt(self, prompt_name, args: dict[str, str]):
        result = await self.session().get_prompt(prompt_name, args)
        return result.messages

    async def read_resource(self, uri: str) -> Any:
        result = await self.session().read_resource(AnyUrl(uri))
        resource = result.contents[0]
        if isinstance(resource, types.TextResourceContents):
            if resource.mimeType == "application/json":
                return json.loads(resource.text)
            return resource.text

    async def cleanup(self):
        await self._exit_stack.aclose()
        self._session = None

    async def __aenter__(self):
        await self.connect()
        return self

    async def __aexit__(self, exc_type, exc_val, exc_tb):
        await self.cleanup()


async def main():
    async with MCPClient(
        command="uv",
        args=["run", "mcp_server.py"],
    ) as _client:
        result = await _client.list_tools()
        print(result)


if __name__ == "__main__":
    if sys.platform == "win32":
        asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
    asyncio.run(main())
'''

with open("mcp_client.py", "w", encoding="utf-8") as f:
    f.write(CLIENT_CODE)

print(f"Wrote: {_os.path.abspath('mcp_client.py')}")
print(f"Size: {_os.path.getsize('mcp_client.py')} bytes")


## §9. 다음 단계 안내

- **S6_04 (Resources)**: 클래스 `MCPClient` 에 메서드 `read_resource` 를 추가하고, URI `docs://documents` 의 목록 형태와 템플릿 형태 `docs://documents/{doc_id}` 두 가지를 모두 호출해 본다.
- **S6_05 (Prompts)**: 메서드 `list_prompts` 와 `get_prompt` 를 추가하고, 서버가 제공하는 프롬프트 `format` 으로 도구 `edit_document` 의 호출 연쇄를 Claude 에게 시킨다.
- **`structural/` 트랙**: 같은 클래스 `MCPClient` 가 KDS 조문 데이터나 Midas 해석 결과를 노출하는 도메인 특화 MCP 서버에 그대로 연결된다.
